# 10 · Error analysis (Step 3.4)

**Runtime: CPU** (Runtime → Change runtime type → **CPU**). No GPU needed: this reads the prediction files already on Drive.

Compares the untrained model (E1) with the three E2 adapters on the same test examples and breaks the mistakes down by question kind, how much the 10 human answerers agreed, photo quality, and number of "unanswerable" votes. Output: counts and rates only (no questions, answers, ids or photos), so it is safe to commit.

Expect about **5 minutes**.

## 1 · Drive and code

In [ ]:
import os
import subprocess

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
RUNS = f"{DRIVE_ROOT}/runs"
OUT = f"{RUNS}/error-analysis"


def sh(cmd):
    """Run a shell command, stream its output, and raise if it fails."""
    proc = subprocess.Popen(
        cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}). Stop here and paste the output above."
        )


drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh(f"pip install -q -e {REPO_DIR}")
os.makedirs(OUT, exist_ok=True)
for run in ("e1-baseline", "e2-answer-lora", "e2-answerability-lora", "e2-quality-lora"):
    print(run, sorted(f for f in os.listdir(f"{RUNS}/{run}") if f.startswith("predictions_")))

**Paste back:** the commit line and the four lines listing prediction files. E1 should list all three tasks; each E2 run its own task.

## 2 · Run the analysis (~2 min)

In [ ]:
sh(
    f"python -m vizwiz.eval.errors --records {DATA}/processed --split test --before {RUNS}/e1-baseline "
    f"--after answer={RUNS}/e2-answer-lora answerability={RUNS}/e2-answerability-lora quality={RUNS}/e2-quality-lora "
    f"--report {OUT}/errors_test.json --markdown {OUT}/errors_test.md"
)
print(open(f"{OUT}/errors_test.md").read())

**Paste back:** nothing needed; upload the two files below instead (they hold the same tables).

## 3 · Flush Drive, then disconnect
Upload back from `MyDrive/vizwiz/runs/error-analysis/`:
- `errors_test.json`
- `errors_test.md`

Then Runtime → Disconnect and delete runtime.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")